In [ ]:
!pip install -q \
    gradio \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    langchain-groq \
    faiss-cpu \
    pypdf \
    sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q gradio langchain langchain-community \
    langchain-text-splitters langchain-huggingface \
    langchain-groq faiss-cpu pypdf sentence-transformers


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import gradio as gr

from getpass import getpass

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 3. GROQ API KEY
# ============================================================

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass(
        "Enter your Groq API key: "
    )


# ============================================================
# 4. EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")


# ============================================================
# 5. LLM
# ============================================================

print("Loading Groq LLM...")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Groq LLM loaded.")


# ============================================================
# 6. RAG PROMPT
# ============================================================

prompt = ChatPromptTemplate.from_template(
    """
You are a strict PDF question-answering assistant.

Your ONLY source of information is the CONTEXT below.

Rules:

1. Answer ONLY using the provided CONTEXT.
2. Do NOT use your pretrained knowledge.
3. Do NOT use outside information.
4. Do NOT guess.
5. Do NOT invent information.
6. If the answer is not available in the CONTEXT, say:

"I don't know based on the uploaded PDF."

7. If the question is unrelated to the PDF, say:

"I don't know based on the uploaded PDF."

8. If only part of the question is supported,
   answer only the supported part.

9. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
)


# ============================================================
# 7. SIMILARITY THRESHOLD
# ============================================================

SIMILARITY_THRESHOLD = 0.25


# ============================================================
# 8. PROCESS PDF
# ============================================================

def process_pdf(pdf_path):

    if pdf_path is None:
        return None, "❌ Please upload a PDF first."

    try:

        print("\n==============================")
        print("PROCESSING PDF")
        print("==============================")

        print("File:", pdf_path)

        # ----------------------------------------------------
        # LOAD PDF
        # ----------------------------------------------------

        loader = PyPDFLoader(pdf_path)

        documents = loader.load()

        print("Pages loaded:", len(documents))


        # ----------------------------------------------------
        # REMOVE EMPTY PAGES
        # ----------------------------------------------------

        documents = [
            doc
            for doc in documents
            if doc.page_content
            and doc.page_content.strip()
        ]

        print(
            "Non-empty pages:",
            len(documents)
        )


        # ----------------------------------------------------
        # CHECK PDF
        # ----------------------------------------------------

        if not documents:

            return (
                None,
                "❌ No readable text found in this PDF. "
                "It may be a scanned PDF requiring OCR."
            )


        # ----------------------------------------------------
        # CHUNKING
        # ----------------------------------------------------

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=150
        )

        chunks = text_splitter.split_documents(
            documents
        )

        print(
            "Chunks created:",
            len(chunks)
        )


        if not chunks:

            return (
                None,
                "❌ No chunks could be created."
            )


        # ----------------------------------------------------
        # FAISS
        # ----------------------------------------------------

        vectorstore = FAISS.from_documents(
            documents=chunks,
            embedding=embeddings,
            distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
        )

        print(
            "Vector database created."
        )


        # ----------------------------------------------------
        # STATUS
        # ----------------------------------------------------

        status = (
            "## ✅ PDF processed successfully\n\n"
            f"**Pages:** {len(documents)}\n\n"
            f"**Chunks:** {len(chunks)}\n\n"
            "You can now ask questions about the PDF."
        )

        return vectorstore, status


    except Exception as e:

        print(
            "ERROR PROCESSING PDF:",
            str(e)
        )

        return (
            None,
            "❌ Error processing PDF:\n\n"
            + str(e)
        )


# ============================================================
# 9. ASK QUESTION
# ============================================================

def ask_question(question, vectorstore):

    if vectorstore is None:

        return (
            "❌ Please upload and process a PDF first."
        )


    if not question or not question.strip():

        return (
            "❌ Please enter a question."
        )


    try:

        print("\n==============================")
        print("QUESTION")
        print("==============================")

        print(question)


        # ----------------------------------------------------
        # RETRIEVE
        # ----------------------------------------------------

        results = vectorstore.similarity_search_with_score(
            question,
            k=4
        )


        # ----------------------------------------------------
        # SHOW SCORES
        # ----------------------------------------------------

        print("\nRetrieval scores:")

        for i, (doc, score) in enumerate(results):

            print(
                f"Chunk {i + 1}: {score:.4f}"
            )

            print(
                "Page:",
                doc.metadata.get(
                    "page",
                    "Unknown"
                )
            )

            print(
                "Text:",
                doc.page_content[:150]
            )

            print("-" * 50)


        # ----------------------------------------------------
        # RELEVANCE FILTER
        # ----------------------------------------------------

        relevant_docs = []

        for doc, score in results:

            if score >= SIMILARITY_THRESHOLD:

                relevant_docs.append(doc)


        print(
            "Relevant chunks:",
            len(relevant_docs)
        )


        # ----------------------------------------------------
        # HALLUCINATION PROTECTION
        # ----------------------------------------------------

        if not relevant_docs:

            return (
                "I don't know based on the uploaded PDF."
            )


        # ----------------------------------------------------
        # BUILD CONTEXT
        # ----------------------------------------------------

        context_parts = []

        for doc in relevant_docs:

            page = doc.metadata.get("page")

            if page is not None:

                page_number = page + 1

                context_parts.append(
                    f"[Page {page_number}]\n"
                    f"{doc.page_content}"
                )

            else:

                context_parts.append(
                    doc.page_content
                )


        context = "\n\n".join(
            context_parts
        )


        # ----------------------------------------------------
        # PROMPT
        # ----------------------------------------------------

        messages = prompt.invoke(
            {
                "context": context,
                "question": question
            }
        )


        # ----------------------------------------------------
        # CALL GROQ
        # ----------------------------------------------------

        print(
            "Calling Groq..."
        )

        response = llm.invoke(
            messages
        )

        answer = response.content


        # ----------------------------------------------------
        # SOURCE PAGES
        # ----------------------------------------------------

        pages = []

        for doc in relevant_docs:

            page = doc.metadata.get("page")

            if page is not None:

                page_number = page + 1

                if page_number not in pages:

                    pages.append(
                        page_number
                    )


        pages.sort()


        # ----------------------------------------------------
        # FINAL ANSWER
        # ----------------------------------------------------

        final_answer = answer

        if pages:

            final_answer += (
                "\n\n---\n\n"
                "📄 **Source Pages:** "
                + ", ".join(
                    f"Page {page}"
                    for page in pages
                )
            )


        return final_answer


    except Exception as e:

        print(
            "QUESTION ERROR:",
            str(e)
        )

        return (
            "❌ Error:\n\n"
            + str(e)
        )


# ============================================================
# 10. GRADIO UI
# ============================================================

with gr.Blocks(
    title="Ravi Teja PDF RAG Chatbot"
) as demo:


    gr.Markdown(
        """
# 📚 PDF RAG Chatbot

Upload a PDF and ask questions based only
on the uploaded document.

### Pipeline

PDF
↓
Text Extraction
↓
Chunking
↓
Embeddings
↓
FAISS
↓
Similarity Search
↓
Relevance Filter
↓
Context
↓
Groq / Llama
↓
Answer + Sources
"""
    )


    # --------------------------------------------------------
    # PDF
    # --------------------------------------------------------

    with gr.Row():

        pdf_input = gr.File(
            label="📄 Upload PDF",
            file_types=[".pdf"],
            type="filepath"
        )

        process_button = gr.Button(
            "⚙️ Process PDF",
            variant="primary"
        )


    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    status = gr.Markdown(
        "Upload a PDF and click **Process PDF**."
    )


    # --------------------------------------------------------
    # VECTORSTORE
    # --------------------------------------------------------

    vectorstore_state = gr.State(
        None
    )


    # --------------------------------------------------------
    # QUESTION
    # --------------------------------------------------------

    question = gr.Textbox(
        label="💬 Ask a question",
        placeholder=(
            "Example: What is the abs() function?"
        ),
        lines=2
    )


    ask_button = gr.Button(
        "🔎 Ask Question",
        variant="primary"
    )


    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    output = gr.Markdown(
        label="Answer"
    )


    # --------------------------------------------------------
    # PROCESS PDF
    # --------------------------------------------------------

    process_button.click(
        fn=process_pdf,
        inputs=pdf_input,
        outputs=[
            vectorstore_state,
            status
        ]
    )


    # --------------------------------------------------------
    # ASK
    # --------------------------------------------------------

    ask_button.click(
        fn=ask_question,
        inputs=[
            question,
            vectorstore_state
        ],
        outputs=output
    )


    # Allow ENTER key
    question.submit(
        fn=ask_question,
        inputs=[
            question,
            vectorstore_state
        ],
        outputs=output
    )


# ============================================================
# 11. LAUNCH
# ============================================================

demo.launch(
    share=True,
    debug=True
)

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.
Loading Groq LLM...
Groq LLM loaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c4984e5d67df7d6370.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



PROCESSING PDF
File: /tmp/gradio/b16a9ea3b5b7344cd16b68e9d3ec5154c4edfe7492388684b364ae60f7f826a7/Python_inbuildfunctions.pdf
Pages loaded: 15
Non-empty pages: 15
Chunks created: 17
Vector database created.

QUESTION
What is the abs() function?

Retrieval scores:
Chunk 1: 0.4265
Page: 1
Text: 1 | P a g e  
 
1. abs() 
The abs() is one of the most popular Python built -in functions, 
which returns the absolute value of a number. A negative v
--------------------------------------------------
Chunk 2: 0.2482
Page: 5
Text: 13. complex() 
complex() function creates a complex number. We have seen this is 
our article on  Python  Numbers . 
>>>  comple x(3) 
(3+0j)
--------------------------------------------------
Chunk 3: 0.2401
Page: 12
Text: 12 | P a g e  
 
>>>  po w(3,4) 
81 
>>>  po w(7,0) 
1 
>>>  po w(7,-1) 
0.14285714285714285  
>>>  po w(7,-2) 
0.02040816326530612  
35. print() 
We 
--------------------------------------------------
Chunk 4: 0.2314
Page: 11
Text: 11 | P a g e  
 